# 08: As tools do Investigator

O Investigator existe para explicar o **porquê** de um número estranho, não para repetir o número.
Para isso ele tem três tools com papéis separados:

- `buscar_fatos_relevantes`: eventos corporativos formais. É a primeira busca, porque é a "confissão oficial" da empresa.
- `buscar_historico_trimestral`: 8 trimestres de lucro, receita e lucro bruto, para ver se a queda é pontual ou uma tendência.
- `buscar_noticias_web`: causas externas (commodities, câmbio, setor). Já apareceu no notebook 07.

Este notebook mostra as duas primeiras com a VALE3, que hoje tem lucro em queda.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Histórico trimestral

Um **trimestre** é o período de 3 meses em que as empresas abertas divulgam resultados. Os números estão em
bilhões de reais.

- **Net Income (lucro líquido):** o que sobra depois de todos os custos, despesas, juros e impostos.
- **Total Revenue (receita):** o total vendido no período.
- **Gross Profit (lucro bruto):** receita menos o custo direto do que foi vendido.

In [2]:
from fluxo_financeiro.tools.investigation import buscar_fatos_relevantes, buscar_historico_trimestral

print(buscar_historico_trimestral.invoke("VALE3"))

2026-11-06 21:38:30,621 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ticker=VALE3.SA


2026-11-06 21:38:31,143 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ok — 7 trimestres


Histórico Trimestral de VALE3.SA (últimos 7 trimestres):

Métrica             2026-09-30    2026-06-30    2026-03-31    2025-12-31    2025-09-30    2025-06-30    2025-03-31    
----------------------------------------------------------------------------------------------------------------------
Net Income                1.25B        1.38B        1.89B       -3.84B        2.69B         N/A          N/A   
Total Revenue            10.60B       10.50B        9.26B       11.06B       10.42B         N/A          N/A   
Gross Profit              2.66B        3.20B        3.08B        4.28B        3.79B         N/A          N/A   



**Como ler a tabela** (a coluna mais à esquerda é o trimestre mais recente):
- **Pontual:** um trimestre fora da curva e os outros normais. Costuma ser um evento único, como uma baixa contábil.
- **Estrutural:** piora trimestre após trimestre. O negócio está perdendo força.
- **Cíclica:** sobe e desce acompanhando um ciclo externo, como o preço do minério de ferro.

Compare o lucro com a receita: se a receita ficou estável e o lucro despencou, o problema está nos custos ou num
evento contábil, não nas vendas.

## 2. Fatos relevantes

- **Fato relevante:** comunicado que a empresa é obrigada a publicar na CVM quando acontece algo que pode mexer no preço da ação (uma venda de ativo, um acordo judicial, uma mudança na diretoria).
- **Impairment:** baixa contábil quando um ativo passa a valer menos do que está registrado no balanço. Derruba o lucro do trimestre sem tirar dinheiro do caixa.
- **Provisão:** dinheiro reservado no balanço para uma perda provável no futuro (por exemplo, uma indenização).

A tool faz uma busca na web focada nesses eventos nos últimos 6 meses.

In [3]:
print(buscar_fatos_relevantes.invoke("VALE3"))

2026-11-06 21:38:31,150 - fluxo_financeiro - INFO - [tool] buscar_fatos_relevantes: ticker=VALE3


2026-11-06 21:38:34,480 - fluxo_financeiro - INFO - [tool] buscar_fatos_relevantes: ok (1135 chars)


Fatos relevantes e eventos corporativos de VALE3:

<conteudo_externo origem="google_search">
Nos últimos seis meses, o desempenho financeiro da Vale (VALE3) foi impactado principalmente pela retração nos preços globais do minério de ferro — reflexo da menor demanda da China —, além de pressões inflacionárias de custos e efeitos cambiais decorrentes da apreciação do real. No balanço do terceiro trimestre de 2026, a companhia reportou um lucro líquido de US$ 1,255 bilhão, o que representou uma queda expressiva de 53% em relação ao mesmo período do ano anterior, acompanhado por uma retração de 20% no Ebitda Proforma.

Apesar da compressão nas margens operacionais do principal segmento, a divisão de metais básicos (VBM) seguiu apresentando melhorias de eficiência e avanços em ativos de cobre e níquel. Em termos de alocação de capital e eventos corporativos extraordinários, a gestão adotou uma postura de maior cautela devido a incertezas macroeconômicas e geopolíticas globais, descartando o

## O que observar

- No histórico, veja se a queda do lucro é de um trimestre só ou de vários: é exatamente o que o Investigator precisa decidir.
- Se o yfinance trouxer menos de 8 trimestres, a tabela mostra quantos vieram; o yfinance também atrasa os trimestres mais recentes.
- Os fatos relevantes chegam dentro de `<conteudo_externo>`, com um cabeçalho fora da tag: o cabeçalho é do sistema, o conteúdo é da web.
- Um impairment encontrado nos fatos e um trimestre isolado no histórico, juntos, apontam para um evento pontual.